# 第11課 - 代理人對代理人 (A2A) 協議


## 設定


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## A2A 協定是什麼？

The **Agent-to-Agent (A2A) 協定** 是一個開放標準，使 AI 代理人能夠溝通、
互相發現，並協作 —— 即使它們建置於不同的框架或由不同服務所託管。

關鍵概念：

- **發現** – 代理人會發佈一張 *代理人卡片*，描述其能力，讓其他代理人（或協調者）更容易找到適合處理任務的專家。
- **訊息傳遞** – 代理人透過共同協定交換結構化訊息，因此一個
  代理人的請求能被另一個代理人理解並完成，而不受其內部
  實作的限制。
- **任務生命週期** – A2A 定義了像 *已提交*、*處理中*、*已完成* 和
  *失敗*，讓協調者完整掌握委派任務的進度。

在本課中，我們透過串接三個專門的旅遊代理人
到一個工作流程中，每個代理人都貢獻其專長並將結果傳遞給下一位。


## 建立專門化的旅遊代理人


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## 透過工作流程的多代理協作

我們將這三個代理連接成一個順序工作流程，模擬 A2A 訊息傳遞：

1. **CurrencyExchangeAgent** 接收使用者的請求並提供貨幣指引。
2. **ActivityPlannerAgent** 接收已豐富的上下文並加入活動建議。
3. **TravelManagerAgent** 綜合兩者輸入成最終的旅遊摘要。


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## 在生產環境中理解 A2A

在生產環境中，A2A 協議可釋放強大的跨服務場景：

| 能力 | 說明 |
|---|---|
| **跨框架互通** | 使用某個框架建立的代理可以將任務委派給任何其他符合 A2A 的框架所建立的代理，從而實現真正的跨組織互通。 |
| **服務邊界** | 代理可以存在於獨立的微服務、雲端區域，甚至不同的組織中，同時仍能無縫協作。 |
| **動態發現** | 協調器可以在執行時查詢 Agent Card 註冊庫，以找到最適合特定子任務的專家。 |
| **串流與推播通知** | A2A 支援 Server-Sent Events (SSE)，用於即時進度更新，並為長時間執行的任務提供推播通知。 |

我們上面建立的工作流程是此模式的一個簡化、同進程版本。在實際部署中，每個代理會暴露一個 HTTP 端點、發佈 Agent Card，並透過 A2A JSON-RPC 協議進行通訊。


## 摘要

在本課程中你學到了:

1. **A2A 協定是什麼** — 一個用於代理間發現, 訊息傳遞,
   與任務管理.
2. **如何建立專門化代理** — 一個貨幣兌換代理, 一個活動規劃代理,
   以及一個旅行管理編排者.
3. **如何將代理串接成工作流程** — 使用 `WorkflowBuilder` 來模擬順序的
   代理之間的訊息傳遞.
4. **A2A 在生產環境中的運作方式** — 使跨框架, 跨服務的協作
   具備動態發現與串流更新.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件由 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們力求準確，但請注意自動翻譯可能包含錯誤或不準確之處。原始語言的文件應視為具權威性的來源。對於重要資訊，建議採用專業人工翻譯。我們不對因使用此翻譯而導致的任何誤解或錯誤詮釋負責。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
